# Síntese dos resultados para a monografia (estágio 15)

Consolida os artefatos dos estágios anteriores em tabelas e figuras de síntese. As tabelas (CSV e LaTeX) cobrem o dataset por fonte, o treinamento e a avaliação por dobra, o agregado micro, a comparação estatística emparelhada e a relevância quantitativa dos mapas de XAI dentro e fora da referência de café. As figuras resumem as curvas de treino, as métricas de avaliação/comparação e a relevância dos mapas Grad-CAM e Attention Rollout. O estágio não retreina nem reavalia: apenas lê artefatos vigentes dos estágios 06–14 e é regenerado quando alguma entrada muda.

## Bootstrap do workspace

O primeiro passo baixa e executa `src/bootstrap.py` (somente stdlib) — necessário porque o `src/` ainda não está disponível para import em uma sessão nova. O bootstrap obtém o repositório público, extrai `src/`, `data/external/` e `requirements-runtime.txt` para o workspace e adiciona o workspace ao `sys.path`. O `reload` garante que reexecuções usem a versão mais recente baixada.

In [ ]:
# Baixa e executa o bootstrap do workspace (etapa prévia ao import de src/).
import importlib
import pathlib
import sys
import urllib.request

BOOTSTRAP_URL = "https://raw.githubusercontent.com/jotap1101/tcc-lab/main/src/bootstrap.py"
pathlib.Path("bootstrap.py").write_bytes(urllib.request.urlopen(BOOTSTRAP_URL).read())
sys.path.insert(0, str(pathlib.Path.cwd()))

# Recarrega o módulo para não reutilizar uma versão antiga em cache no kernel.
bootstrap = importlib.import_module("bootstrap")
importlib.reload(bootstrap)

workspace = bootstrap.bootstrap_workspace()
print(f"Workspace: {workspace}")

## Dependências pinadas

Garante as versões fixadas em `requirements-runtime.txt`, o mesmo conjunto nas duas plataformas. Quando uma biblioteca com extensão compilada já importada (numpy, scipy, torch, entre outras) diverge da versão pinada, o kernel é reiniciado automaticamente para evitar inconsistência de ABI; nesse caso, o notebook é reexecutado a partir da primeira célula.

In [ ]:
# Garante as versões pinadas e reinicia o kernel quando a ABI carregada divergir.
from src.setup import install_runtime_requirements

requirements = pathlib.Path(workspace) / "requirements-runtime.txt"
install_runtime_requirements(requirements)

## Pacote compartilhado, plataforma e armazenamento

Importa o pacote `src/` (já entregue pelo bootstrap) e identifica a plataforma pela abstração em `src/io.py`. Em seguida garante a raiz `MyDrive/tcc/` e resolve os caminhos de armazenamento definidos em `src/config.yaml`, incluindo dados, modelos, artefatos de avaliação, tabelas de síntese e figuras. A fonte sintetizada é a mesma escolhida manualmente na fase de ground truth.

In [ ]:
# Resolve o contexto de runtime e os caminhos canônicos da síntese.
from src.data.source_decision import chosen_source
from src.setup import initialize
from src.synthesis import metadata_path, summary_path, synthesis_dir

ctx = initialize(workspace)
platform = ctx.platform
storage_paths = ctx.storage_paths
config = ctx.config
source_name = chosen_source()
print(f"Plataforma: {platform}")
print(f"Fonte de treinamento/avaliação (decisão manual): {source_name}")
print(f"Diretório da síntese: {synthesis_dir(storage_paths, source_name)}")
print(f"Resumo: {summary_path(storage_paths, source_name)}")
print(f"Metadata: {metadata_path(storage_paths, source_name)}")

## Reprodutibilidade

Fixa as sementes de python/numpy/torch/cuda e habilita as flags determinísticas do PyTorch, além de registrar as versões do ambiente. A síntese apenas lê artefatos e renderiza tabelas/figuras em CPU; não há treinamento nem inferência de modelos neste estágio.

In [ ]:
# Fixa sementes, flags determinísticas e registra o ambiente.
from src.utils import log_environment, setup_reproducibility

setup_reproducibility(config)
log_environment()

## Dependências dos estágios 06–14

Verifica o manifesto da fonte escolhida, os resumos vigentes do estágio 11 para U-Net e SegFormer, a comparação do estágio 12, os artefatos de treino dos estágios 09/10 e, quando presentes, os metadados/mapas de XAI dos estágios 13/14. O XAI é opcional na síntese; metadados presentes precisam estar completos e vigentes.

In [ ]:
# Valida as dependências e a vigência dos artefatos usados na síntese.
from src import comparative
from src.data.patch_generation import manifest_path
from src.evaluation import evaluation_summary_path
from src.synthesis import validate_inputs
from src.utils import check_dependencies

check_dependencies(
    {
        "Manifesto (estágio 06)": manifest_path(storage_paths, source_name),
        "Resumo da U-Net (estágio 11)": evaluation_summary_path(
            storage_paths, "unet", source_name
        ),
        "Resumo do SegFormer (estágio 11)": evaluation_summary_path(
            storage_paths, "segformer", source_name
        ),
        "Comparação (estágio 12)": comparative.comparison_path(storage_paths, source_name),
    }
)
source_name, summaries, comparison, xai_models = validate_inputs(storage_paths, source_name)
print(f"Dobras por modelo: {len(summaries['unet']['per_fold'])}")
print(f"Patches avaliados: {comparison['n_patches']}")
print(f"XAI incluído: {xai_models if xai_models else 'nenhum'}")

## Geração das tabelas e figuras

Gera as tabelas de dataset, treinamento, avaliação por dobra e agregada, comparação estatística e relevância de XAI, além das figuras de curvas de treino, visão geral da avaliação/comparação e relevância quantitativa dos mapas. Os artefatos são persistidos no Drive e reutilizados apenas enquanto o fingerprint de todas as entradas coincide.

In [ ]:
# Gera (ou reutiliza) as tabelas e figuras de síntese da fonte escolhida.
from src.synthesis import run_synthesis

synthesis_result = run_synthesis(storage_paths, source_name=source_name)
print(f"Artefatos reutilizados: {synthesis_result.skipped}")
print(f"Tabelas: {len(synthesis_result.tables)} | Figuras: {len(synthesis_result.figures)}")

## Tabelas consolidadas

Apresenta as principais tabelas: resumo do dataset por fonte, métricas agregadas de avaliação, estatísticas emparelhadas (SegFormer − U-Net) e relevância dos mapas de XAI. As versões completas em CSV e LaTeX ficam no diretório de síntese.

In [ ]:
# Exibe as principais tabelas consolidadas da síntese.
from IPython.display import display

for name in ("dataset", "evaluation_aggregate", "comparative_paired", "xai"):
    if name in synthesis_result.tables_data:
        print(f"Tabela: {name}")
        display(synthesis_result.tables_data[name])

## Figuras de síntese

Exibe as figuras geradas: curvas de treino por dobra, visão geral da avaliação e da comparação e, quando disponível, a relevância quantitativa dos mapas de XAI.

In [ ]:
# Exibe as figuras de síntese persistidas no Drive.
from IPython.display import Image, display

for name, path in synthesis_result.figures.items():
    print(f"Figura: {name}")
    display(Image(filename=str(path)))

## Resumo da etapa

Exibe os principais números da síntese, o inventário de tabelas/figuras e os caminhos dos artefatos consolidados.

In [ ]:
# Exibe o resumo do estágio e os caminhos dos artefatos de síntese.
from src.utils import print_summary

unet_metrics = synthesis_result.summary["models"]["unet"]["aggregate"]
segformer_metrics = synthesis_result.summary["models"]["segformer"]["aggregate"]
summary = {
    "Fonte": source_name,
    "Patches da fonte escolhida": synthesis_result.summary["n_patches_chosen"],
    "IoU U-Net (micro)": round(float(unet_metrics["iou"]), 4),
    "IoU SegFormer (micro)": round(float(segformer_metrics["iou"]), 4),
    "Delta IoU (SegFormer − U-Net)": round(
        float(synthesis_result.summary["aggregate_delta"]["iou"]), 4
    ),
    "XAI incluído": synthesis_result.summary["xai"]["included"],
    "Tabelas": str(synthesis_dir(storage_paths, source_name) / "tables"),
    "Figuras": str(storage_paths["artifacts_figures"]),
    "Resumo": str(synthesis_result.summary_path),
    "Metadata": str(synthesis_result.metadata_path),
}
print_summary(summary, "15")